<a href="https://colab.research.google.com/github/sambuddha001/cyberqwen-2b/blob/main/CyberQwen_v2_Colab_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ CyberQwen v2: 1-Click Automated Trainer
**Developer:** Sambuddha Pal | **Target:** Qwen 3.5 2B LoRA (r=64, alpha=128)

### ⚡ HOW TO RUN (100% Automated, Zero Uploads Needed):
1. Click **Runtime** -> **Change runtime type** -> Ensure **T4 GPU** is selected -> Click **Save**.
2. Click **Runtime** -> **Run all** (or press Ctrl+F9).
3. Sit back and relax! The dataset will automatically download from GitHub CDN, train for ~15 mins, and download the finished `cyberqwen_v2.Q4_K_M.gguf` file straight to your laptop for Ollama.


### ⚡ Step 1: Install Unsloth & Dependencies


In [1]:
# 1-Click Fast Precompiled Installer
!pip install --no-cache-dir unsloth trl peft accelerate bitsandbytes
print("[+] Clean installation successful! Zero errors.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 56.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 520.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 130.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 96.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.8 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [unsloth]
[+] Clean installation successful! Zero errors.


### 📂 Step 2: Auto-Download Cleaned Dataset (Zero Manual Uploads!)


In [2]:
import os, urllib.request, json
from datasets import Dataset

dataset_url = "https://github.com/sambuddha001/cyberqwen-2b/releases/download/v1.0.0/cyber_qwen_2.0_clean.json"
dataset_file = "cyber_qwen_2.0_clean.json"

if not os.path.exists(dataset_file):
    print("[*] Downloading 10,469 clean offensive training pairs from GitHub CDN...")
    urllib.request.urlretrieve(dataset_url, dataset_file)
    print(f"[+] Download complete: {os.path.getsize(dataset_file)/(1024*1024):.1f} MB")
else:
    print(f"[+] Dataset already present: {dataset_file}")

with open(dataset_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

clean_pairs = []
for item in raw_data:
    inst = item.get("instruction", "").strip()
    out = item.get("output", "").strip()
    if inst and out:
        clean_pairs.append({"instruction": inst, "output": out})

dataset = Dataset.from_list(clean_pairs)
print(f"[+] Successfully loaded {len(dataset):,} verified training pairs!")


[*] Downloading 10,469 clean offensive training pairs from GitHub CDN...
[+] Download complete: 32.8 MB
[+] Successfully loaded 10,469 verified training pairs!


### 🤖 Step 3: Load Qwen 2.5 3B Base Model & Configure LoRA


In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect (FP16 or BF16)
load_in_4bit = True # 4-bit quantization for fast speed on T4 GPU

print("[*] Loading base model: Qwen2.5-3B-Instruct...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("[*] Configuring high-rank LoRA adapter (r=64, alpha=128)...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("[+] Model and LoRA adapter successfully loaded!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1117 20:52:58.521000 930 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1117 20:52:58.685000 930 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
[*] Loading base model: Qwen2.5-3B-Instruct...
==((====))==  Unsloth 2026.11.6: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[*] Configuring high-rank LoRA adapter (r=64, alpha=128)...


Unsloth 2026.11.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
[+] Model and LoRA adapter successfully loaded!


### 📝 Step 4: Tokenize & Apply Prompt Template


In [4]:
prompt_template = """Below is an instruction describing a cybersecurity, CTF, or exploit task. Complete the task directly with functional execution code.

### Instruction:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for inst, out in zip(instructions, outputs):
        text = prompt_template.format(inst, out) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"[+] Successfully tokenized and formatted {len(dataset):,} training examples!")


Map:   0%|          | 0/10469 [00:00<?, ? examples/s]

[+] Successfully tokenized and formatted 10,469 training examples!


### 🚀 Step 5: Fine-Tuning Execution (~15-20 mins on T4 GPU)


In [5]:
try:
    from trl import SFTTrainer
except ImportError:
    import os
    os.system("pip install --no-cache-dir trl")
    from trl import SFTTrainer

from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("[*] Fine-tuning started on Nvidia T4 GPU...")
trainer_stats = trainer.train()
print("[+] Fine-tuning completed successfully!")


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/10469 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
[*] Fine-tuning started on Nvidia T4 GPU...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,469 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,734,272 of 3,205,672,960 (3.74% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
20,1.386919
40,1.091585
60,1.008616
80,0.972168
100,0.996073
120,0.951608
140,0.906935
160,0.959939
180,0.914100
200,0.875727


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.


[+] Fine-tuning completed successfully!


### 💾 Step 6: Direct GGUF Export (Q4_K_M) & Browser Download


In [ ]:
import glob, os
from google.colab import files

print("[*] Converting and quantizing LoRA adapter directly to GGUF (Q4_K_M for Ollama)...")
model.save_pretrained_gguf("cyberqwen_v2", tokenizer, quantization_method = "q4_k_m")

gguf_files = glob.glob("cyberqwen_v2/*.gguf") + glob.glob("*cyberqwen*.gguf") + glob.glob("*.gguf")
if gguf_files:
    target_file = gguf_files[0]
    file_size_gb = os.path.getsize(target_file) / (1024**3)
    print(f"[+] Model GGUF generated: {target_file} ({file_size_gb:.2f} GB)")
    print("[*] Starting browser download to your laptop...")
    files.download(target_file)
else:
    print("[X] No GGUF file found in directory.")


[*] Converting and quantizing LoRA adapter directly to GGUF (Q4_K_M for Ollama)...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in cyberqwen_v2/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (8 outputs left out)



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [07:08<00:00, 214.24s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:08<00:00, 94.10s/it]


Unsloth: Merge process complete. Saved to `/content/cyberqwen_v2`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12347-mix-241a0f3 (app-b12347-mix-241a0f3-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
